## Second test of VA in chamber

In [ ]:
# Imports
import numpy as np
import time
from datetime import datetime
import os

from mcg_lab_vna_control import connect_vna
from mcg_lab_triaxial_positioner_control import TriaxialPositioner

# Transparent conection: the library knows witch driver to use
vna = connect_vna(brand='keysight')


In [ ]:
def time_sleep_sweep_time(n_sweeps_multiplier):
    sweep_time = vna.query_sweep_time(verbose=False)
    time.sleep(n_sweeps_multiplier*sweep_time + 1)  # Wait for the sweep to complete

def time_now():
    return f"[{datetime.now().strftime('%H:%M:%S')}]"

def get_timestamp():
    return f"{datetime.now().strftime('%H:%M:%S')}"

In [ ]:
########## Parameters ##########
# # Triaxial Positioner Port for TCP Server (with Raspi)
# CONN_STR = "tcp://192.168.100.2:5000"

# VNA Parameters
ifbw = 10e3
pow_dbm_list = [-15, 0, 0, 0] # Power levels for each port (in dBm)
n_sweeps = 10
n_average = 1

sparam_list = ['S21', 'S31', 'S41', 'S22', 'S33', 'S44', 'S32', 'S42', 'S23', 'S43', 'S24', 'S34']
sparam_list_3 = sparam_list[0:3]
sparam_list_6 = sparam_list[0:6]
extra_sparam_list = sparam_list[3:]
extra_display_sparam_list = [False, False, False, False, False, False, False, False, False]
n_sparams = len(sparam_list)
trace_list_all = [v+1 for v in range(n_sparams)] # List of all traces to measure
trace_list_bistatic = [1, 2, 3] # List of bistatic traces to measure ALWAYS

f_start_list = [3.79996e9, 6.79984e9, 7.29984e9, 10.04984e9, 12.79984e9, 14.79984e9] # Segmented f_start list (in Hz)
f_stop_list = [3.90004e9, 7.20016e9, 7.70016e9, 10.45016e9, 13.20016e9, 15.20016e9] # Segmented f_stop list (in Hz)
points_list = [279, 557, 557, 557, 557, 557] # Segmented points list

# Folder name for saving measurement files
folder_name = 'data/VA_chamber_test_2'

#################################

if not os.path.exists(folder_name):
    os.makedirs(folder_name, exist_ok=True)

conf_params = {
    'ifbw': ifbw,
    'pow_dbm_list': pow_dbm_list,
    'n_sweeps': n_sweeps,
    'n_average': n_average,
    'sparam_list': sparam_list,
    'f_start_list': f_start_list,
    'f_stop_list': f_stop_list,
    'points_list': points_list
}

### Calibrate the VNA for Segmented Measurements

In [ ]:
vna.preset()

vna.configure_multiple_segmented_sparams(sparam_list=sparam_list_6, f_start_list=f_start_list, f_stop_list=f_stop_list, points_list=points_list, ifbw=ifbw, pow_dbm_list=pow_dbm_list)
vna_freqs = vna.get_freq_vector()

time_sleep_sweep_time(n_sweeps_multiplier = 6)
vna.auto_scale_all()
vna.run_cont()

Measurement time 0.304061149172 seconds


### Calibration

#### Port 1:

In [35]:
vna.init_SOL_calibration(port = 1, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 1: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 1"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 1"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 1"


In [38]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(1)

Measurement of SOL calibrator step 1 completed


In [39]:
# Apply calibration
vna.apply_calibration()

#### Port 2:

In [40]:
vna.init_SOL_calibration(port = 2, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 2: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 2"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 2"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 2"


In [43]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(3)

Measurement of SOL calibrator step 3 completed


In [44]:
# Apply calibration
vna.apply_calibration()

#### Port 3:

In [12]:
vna.init_SOL_calibration(port = 3, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 3: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 3"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 3"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 3"


In [15]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(3)

Measurement of SOL calibrator step 3 completed


In [16]:
# Apply calibration
vna.apply_calibration()

### Measure S21/S31/S41 of Through for back-to-back calibration

In [ ]:
# Measureme S21 of Through
sparam = vna.measure_trace_sparam(trace=1)
vna.auto_scale_all()
np.save(folder_name + '/S21_b2b_cal.npy', sparam)

In [ ]:
# Measureme S31 of Through
sparam = vna.measure_trace_sparam(trace=2)
vna.auto_scale_all()
np.save(folder_name + '/S31_b2b_cal.npy', sparam)

In [ ]:
# Measureme S41 of Through
sparam = vna.measure_trace_sparam(trace=3)
vna.auto_scale_all()
np.save(folder_name + '/S41_b2b_cal.npy', sparam)

### Basic VNA Config to see Time-Response of the Channel

In [ ]:
vna.pseudo_preset()

vna.configure_multiple_sparams(sparam_list = sparam_list_6, f_start = 10e9, f_stop = 15e9, points = 5001, ifbw = 10e3, pow_dbm_list=pow_dbm_list)

for idx in range(sparam_list):
    vna.set_timedomain(trace = idx)

vna.set_Xscale(xmin = 160, xmax = 190, mode = 'dist', trace = 1)
vna.set_Xscale(xmin = 160, xmax = 190, mode = 'dist', trace = 2)
vna.set_Xscale(xmin = 160, xmax = 190, mode = 'dist', trace = 3)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 4)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 5)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 6)

time_sleep_sweep_time(n_sweeps_multiplier = 6)
vna.auto_scale_all()
vna.run_cont()

Measurement time 0.502209354199 seconds


### Return to Segmented Configuration to Measure

In [ ]:
vna.pseudo_preset()

vna.configure_multiple_segmented_sparams(sparam_list=sparam_list_3, f_start_list=f_start_list, f_stop_list=f_stop_list, points_list=points_list, ifbw=ifbw, pow_dbm_list=pow_dbm_list)
vna.calibration_on()

vna_freqs = vna.get_freq_vector()

time_sleep_sweep_time(n_sweeps_multiplier = 3)
vna.auto_scale_all()
vna.run_cont()

Measurement time 0.304061149172 seconds


### VA Grid Measurement Loop

In [ ]:
########## Parameters ##########

setup_number = 1 # setup number (to have different setup trials)

va_vert_pos = 0 # 0 | 1, idx position in vertical of antenna of the VA 
va_Nh = 10 # VA horizontal (Nh) positions
va_Nv = 1 # VA vertical (Nv) positions
va_deltah = 10.0 # mm of Delta H
va_deltav = 10.0 # mm of Delta V

bs_params = {
    'ant_coord': [39.4809805556, -0.3382888889], # [latitude, longitude] https://latlongdata.com/lat-long-converter/
    'ant_height': 1.4, # height of the antenna in meters
    'ant_orientation': [90, 0], # [elevation, azimuth] == [theta, phi] degrees. Theta in range [0, 180], Phi in range [-180, 180]
    'v_pol_vector': [0, 0, 1], # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used
    'h_pol_vector': None, # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used 
    'ant_type': "omnidirectional", # omnidirectional / power-log / horn / isotropic / ...
    'ant_name': "QOM-ST-2-18-S-SG-R",  # antenna name
    'num_rows': va_Nv, # number of rows (int)
    'num_cols': va_Nh, # number of rows (int)
    'vertical_spacing': va_deltav, # elements' vertical spacing (float) in mm
    'horizontal_spacing': va_deltah # elements' horizontal spacing (float) in mm
}

ue0_params = {
    'ant_coord': [39.4809805556, -0.3382888889], # [latitude, longitude] https://latlongdata.com/lat-long-converter/
    'ant_height': 1.4, # height of the antenna in meters
    'ant_orientation': [90, 0], # [elevation, azimuth] == [theta, phi] degrees. Theta in range [0, 180], Phi in range [-180, 180]
    'v_pol_vector': [0, 0, 1], # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used
    'h_pol_vector': None, # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used 
    'ant_type': "power-log", # omnidirectional / power-log / horn / isotropic / ...
    'ant_name': "Aaronia-PowerLOG-70180",  # antenna name
    'num_rows': 1, # number of rows (int)
    'num_cols': 1, # number of rows (int)
    'vertical_spacing': None, # elements' vertical spacing (float) in mm
    'horizontal_spacing': None # elements' horizontal spacing (float) in mm
}

ue1_params = {
    'ant_coord': [39.4809805556, -0.3382888889], # [latitude, longitude] https://latlongdata.com/lat-long-converter/
    'ant_height': 1.4, # height of the antenna in meters
    'ant_orientation': [90, 0], # [elevation, azimuth] == [theta, phi] degrees. Theta in range [0, 180], Phi in range [-180, 180]
    'v_pol_vector': [0, 0, 1], # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used
    'h_pol_vector': None, # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used 
    'ant_type': "power-log", # omnidirectional / power-log / horn / isotropic / ...
    'ant_name': "Aaronia-PowerLOG-70180",  # antenna name
    'num_rows': 1, # number of rows (int)
    'num_cols': 1, # number of rows (int)
    'vertical_spacing': None, # elements' vertical spacing (float) in mm
    'horizontal_spacing': None # elements' horizontal spacing (float) in mm
}

ue2_params = {
    'ant_coord': None, # [latitude, longitude] https://latlongdata.com/lat-long-converter/
    'ant_height': None, # height of the antenna in meters
    'ant_direction': None, # [elevation, azimuth] == [theta, phi] degrees. Theta in range [0, 180], Phi in range [-180, 180]
    'v_pol_vector': None, # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used
    'h_pol_vector': None, # v[x, y, z] director vector of the polarization. Can be of value None if that pol is not used 
    'ant_type': None, # omnidirectional / power-log / horn / isotropic / ...
    'ant_name': None,  # antenna name
    'num_rows': 1, # number of rows (int)
    'num_cols': 1, # number of rows (int)
    'vertical_spacing': None, # elements' vertical spacing (float) in mm
    'horizontal_spacing': None # elements' horizontal spacing (float) in mm
}

#################################

In [ ]:
# Configure measurement VNA params
freq_vector = vna.get_freq_vector()
freq_points =  len(freq_vector)
timestamp_list = []

if n_average == 1:  # No average
    vna.set_sweeps(sweeps=1)
    vna.unset_average()
    n_sweeps_loop = n_sweeps
    row_sparams = np.zeros((va_Nh, n_sweeps, len(trace_list_all), freq_points), dtype=np.complex64)
else:               # Average
    vna.set_sweeps(sweeps=n_sweeps)
    vna.set_average(avg_sweeps=n_average)
    n_sweeps_loop = 1
    row_sparams = np.zeros((va_Nh, 1, len(trace_list_all), freq_points), dtype=np.complex64)



In [ ]:
############### UPDATE VALUES OF THE LOOP ###############

step_i = 0
step_j = 0

#########################################################

for sweep in range(n_sweeps):
    # Measure and collect sparams
    if step_i == 0 or step_i == va_Nh - 1:
        # Add the extra sparams
        vna.add_multiple_sparams(sparam_list=extra_sparam_list, display_sparam_list = extra_display_sparam_list)

        # Measure all measurements
        sparams = vna.measure_multiple_traces_sparams(trace_list = trace_list_all)

        # Save the sparams in the row file
        row_sparams[step_i, sweep, :, :] = sparams
        timestamp_list.append(get_timestamp())

        # Delete the extra sparams
        vna.delete_multiple_sparams(trace_list=trace_list_all[3:])

    else:
        # Only bistatic measurements (S21, S31, S41)
        sparams = vna.measure_multiple_traces_sparams(trace_list = trace_list_bistatic)

        # Save the sparams in the row file
        row_sparams[step_i, sweep, 0:3, :] = sparams

        timestamp_list.append(get_timestamp())

# Save file for each row
if step_i == va_Nh - 1: # last horizontal position of the row
    # Current Row Filename
    filename_row = f"{folder_name}/setup_{setup_number}_nv_{step_j}"
    np.savez(filename_row, 
        row_sparams=row_sparams, 
        conf_params=conf_params, 
        timestamp_list=timestamp_list, 
        bs_params=bs_params, 
        ue0_params=ue0_params, 
        ue1_params=ue1_params, 
        ue2_params=ue2_params)
    time.sleep(1)